In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

absa_train = load_jsonl(absa_train_path)
absa_dev = load_jsonl(absa_dev_path)
absa_test = load_jsonl(absa_test_path)


In [3]:
ABSA_POLARITIES = {"positive", "negative", "neutral", "conflict"}

def get_allowed_categories(rows):
    cats = set()
    for r in rows:
        for op in r["labels"]:
            cats.add(op["category"])
    return cats
ALLOWED_CATEGORIES = get_allowed_categories(absa_train)


In [4]:
def build_absa_prompt(text, allowed_categories):
    cats_str = ", ".join(sorted(allowed_categories))

    header = (
        "أنت نظام لاستخراج مشاعر الجوانب (Aspect-Based Sentiment) من مراجعات الفنادق باللغة العربية.\n\n"
        "المهمة:\n"
        "استخرج جميع الآراء (opinions) الموجودة في النص.\n"
        "كل رأي يجب أن يكون زوجاً من:\n"
        "- category: جانب محدد بصيغة مثل HOTEL#CLEANLINESS\n"
        "- polarity: واحدة من {positive, negative, neutral, conflict}\n\n"
        "قيود مهمة جداً:\n"
        f"- استخدم فقط الفئات التالية ولا تستطيع ان تكتب اكثر من 8 (categories): {cats_str}\n"
        "صيغة الإخراج بالضبط:\n"
        '{"labels":[{"category":"...","polarity":"..."}, ...]}\n'
        "لا تكتب ولا تكتب اي شيء آخر '''json فقط اخرج بصياغ المكتوب اعلى\n"
        "- إذا لم تجد أي آراء، أرجع: {\"labels\":[]}\n"
        "لا تخمّن. لا تضف أي رأي إلا إذا كان واضحًا ومذكورًا في النص."
        "أخرج بحد أقصى 8 آراء. إذا زادت، اختر الأوضح فقط."
    )

    return f"{header}\n\nالنص:\n{text}\n"

def to_sft_example(r):
  labels=[]
  for op in r["labels"]:
    labels.append({"category": op["category"], "polarity": op["polarity"]})
  prompt = build_absa_prompt(r["text"], ALLOWED_CATEGORIES)
  return {"text": prompt + json.dumps({"labels": labels}, ensure_ascii=False)}


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
import torch
model_path = "drive/MyDrive/FYP/ALLAM-7B"

def load_new_peft_model():
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.config.use_cache = False
    #Attach Lora
    lora = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=4,
        lora_alpha=8,
        lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    )
    model = get_peft_model(model, lora)
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    return model, tokenizer


In [6]:
#model.print_trainable_parameters()


In [7]:
# from datasets import Dataset
# from trl import SFTTrainer
# from transformers import TrainingArguments

# train_ds = Dataset.from_list(train_small_sft)
# dev_ds = Dataset.from_list(dev_sft)

# args = TrainingArguments(
#     output_dir="/content/drive/MyDrive/FYP/slsa/allam_lora_slsa_HPT",
#     per_device_train_batch_size=2,
#     per_device_eval_batch_size=2,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,
#     num_train_epochs=3,
#     logging_steps=50,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     fp16=True,
#     report_to="none",
# )

# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=train_ds,
#     eval_dataset=dev_ds,
#     dataset_text_field="text",
#     max_seq_length=256,
#     args=args,
# )

# trainer.train()


# EVAL FROM NOTEBOOK 2.1

In [8]:
def format_absa_prediction(obj, allowed_categories):
    if not isinstance(obj, dict) or "labels" not in obj:
        return None

    labels = obj["labels"]
    if not isinstance(labels, list):
        return None

    allowed_set = set(allowed_categories)
    out = set()

    for label in labels:
        if not isinstance(label, dict):
            continue
        cat = label.get("category")
        pol = label.get("polarity")
        if isinstance(cat, str): cat = cat.strip()
        if isinstance(pol, str): pol = pol.strip()

        if cat in allowed_set and pol in ABSA_POLARITIES:
            out.add((cat, pol))

    return out

In [9]:
import os, json, re
def extract_first_json(text: str):
    if text is None:
        return None
    text = text.strip()
    m = re.search(r"\{.*\}", text, flags=re.S)
    if not m:
        return None
    block = m.group(0).strip()
    try:
        return json.loads(block)
    except json.JSONDecodeError:
        return None


In [10]:
def generate_chat(model, tokenizer, user_content: str, max_new_tokens: int):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,      # deterministic
            temperature=None,     # ignored when do_sample=False
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )
    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

In [11]:
import re, json
from sklearn.metrics import f1_score, accuracy_score
from tqdm import tqdm

def f1_micro(y_true, y_pred):
    tp = fp = fn = 0
    for true, pred in zip(y_true, y_pred):
        tp += len(true & pred)
        fp += len(pred - true)
        fn += len(true - pred)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2*precision*recall)/(precision+recall) if (precision+recall) else 0.0
    return precision, recall, f1

In [12]:
def eval_absa_prompting(model,tokenizer, rows, allowed_categories):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        prompt = build_absa_prompt(r["text"], allowed_categories)
        print("\nText:", r["text"])
        raw = generate_chat(model, tokenizer, prompt, max_new_tokens=1000)
        print("\nRaw:", raw)
        block = extract_first_json(raw)
        pred = format_absa_prediction(block, allowed_categories)

        if pred is None:
            bad += 1
            pred = set()

        true = set()
        for x in r["labels"]:
            true.add((x["category"], x["polarity"]))
        y_true.append(true)
        y_pred.append(pred)

        print("True:", true)
        print("Pred:", pred)

    p, rc, f1 = f1_micro(y_true, y_pred)
    return {"p": p, "r": rc, "micro_f1": f1, "invalid_json": bad, "n": len(rows)}


In [13]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 9.9 MB/s eta 0:00:00


In [14]:
import os

OUT_BASE   = "/content/drive/MyDrive/FYP/absa/allam_lora_absa_peft"

In [15]:
from datasets import Dataset
train_sft = [to_sft_example(r) for r in absa_train]
train_ds = Dataset.from_list(train_sft)

In [16]:
import gc
from trl import SFTTrainer
from transformers import TrainingArguments


In [17]:
def formatting_func(example):
    return example["text"]


In [18]:
def train_one_run(lr, epochs):
  #Load Model
  model, tokenizer = load_new_peft_model()
  #Training Config
  args = TrainingArguments(
        output_dir=OUT_BASE,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=lr,
        num_train_epochs=epochs,
        logging_steps=50,
        save_strategy="no",
        fp16=True,
        report_to="none",
  )
  #Train
  trainer = SFTTrainer(
        model=model,
        processing_class=tokenizer,
        train_dataset=train_ds,
        formatting_func=formatting_func,
        args=args,
  )
  trainer.train()
  adapter_dir = os.path.join(OUT_BASE, "adapter")
  os.makedirs(adapter_dir, exist_ok=True)
  trainer.model.save_pretrained(adapter_dir)
  #Evaluate
  metrics = eval_absa_prompting(trainer.model,tokenizer, absa_test, ALLOWED_CATEGORIES)
  #Log metrics
  results ={
        "precision": metrics["p"],
        "recall": metrics["r"],
        "micro-f1": metrics["micro_f1"],
        "invalid_json": metrics["invalid_json"],
        "n": metrics["n"],
    }
  #save_metrics
  results_dir = os.path.join(OUT_BASE, "results")
  os.makedirs(results_dir, exist_ok=True)
  with open(os.path.join(results_dir, "results.json"), "w") as f:
      json.dump(results, f, indent=4)
  del trainer
  del model
  torch.cuda.empty_cache()
  gc.collect()

  return metrics


In [19]:
absa_peft_eval = train_one_run(lr=5e-4, epochs=3)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Applying formatting function to train dataset:   0%|          | 0/1564 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1564 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1564 [00:00<?, ? examples/s]

...

 42%|████▏     | 192/452 [27:00<38:30,  8.89s/it]


Raw: {"labels": [{"category": "HOTEL#CLEANLINESS", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#GENERAL", "polarity": "negative"}, {"category": "FOOD_DRINKS#QUALITY", "polarity": "negative"}, {"category": "SERVICE#GENERAL", "polarity": "negative"}, {"category": "HOTEL#GENERAL", "polarity": "negative"}]}
True: {('SERVICE#GENERAL', 'negative'), ('HOTEL#CLEANLINESS', 'negative'), ('HOTEL#GENERAL', 'negative'), ('ROOMS_AMENITIES#MISCELLANEOUS', 'negative'), ('FOOD_DRINKS#MISCELLANEOUS', 'negative')}
Pred: {('SERVICE#GENERAL', 'negative'), ('HOTEL#CLEANLINESS', 'negative'), ('HOTEL#GENERAL', 'negative'), ('ROOMS_AMENITIES#GENERAL', 'negative'), ('FOOD_DRINKS#QUALITY', 'negative')}

Text: تجربة متواضعة الي حد ما !!! الفندق جميل من حيث الاثاث والغرف والنظافة المطعم لايرقي لمستوي نجمة واحدة معاملة موظفي الاستقبال سيئة للغاية حمام السباحة جمبل ونظيف الشاطئ متسخ وسيى للغاية وملئ بالطحالب والزلط الحمام الملحق بالغرفة نظيف ولكن لايوجد شامبو ولا لوشن والمياة السخنة تاتي علي فترات قنوات ا

 42%|████▏     | 192/452 [28:29<38:34,  8.90s/it]


Raw: {"labels": [{"category": "ROOMS#GENERAL", "polarity": "positive"}, {"category": "ROOMS#CLEANLINESS", "polarity": "positive"}, {"category": "FACILITIES#GENERAL", "polarity": "positive"}, {"category": "SERVICE#GENERAL", "polarity": "negative"}, {"category": "FACILITIES#CLEANLINESS", "polarity": "positive"}, {"category": "FACILITIES#DESIGN_FEATURES", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#GENERAL", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#QUALITY", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#PRICES", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#MISCELLANEOUS", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#CLEANLINESS", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#GENERAL", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#MISCELLANEOUS", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#QUALITY", "polarity": "negative"}, {"category": "ROOMS_AMENITIES#PRICES", "polarity": "negative"}, {"category": "ROOMS_

JSONDecodeError: Unterminated string starting at: line 1 column 2658 (char 2657)

### Crashed after during eval after fine tuned so will load the adapters and re-evaluate

In [13]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

In [15]:
adapter_dir = "/content/drive/MyDrive/FYP/absa/allam_lora_absa_peft/adapter"
model_path = "/content/drive/MyDrive/FYP/ALLAM-7B"

tokenizer = AutoTokenizer.from_pretrained(model_path)
base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, adapter_dir)


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [16]:
def eval_absa_with_model(model, tokenizer, rows, allowed_categories):
    bad = 0
    y_pred, y_true = [], []

    for r in tqdm(rows):
        prompt = build_absa_prompt(r["text"], allowed_categories)
        raw = generate_chat(model, tokenizer, prompt, max_new_tokens=350)

        block = extract_first_json(raw)
        pred = format_absa_prediction(block, allowed_categories)
        if pred is None:
            bad += 1
            pred = set()

        true = set((x["category"], x["polarity"]) for x in r["labels"])
        y_true.append(true)
        y_pred.append(pred)

    p, rc, f1 = f1_micro(y_true, y_pred)
    return {"p": p, "r": rc, "micro_f1": f1, "invalid_json": bad, "n": len(rows)}


In [17]:
absa_peft_test = eval_absa_with_model(model, tokenizer, absa_test, ALLOWED_CATEGORIES)

100%|██████████| 452/452 [1:00:26<00:00,  8.02s/it]


In [21]:
print(absa_peft_test)

{'p': 0.6770276833414279, 'r': 0.6459684893419834, 'micro_f1': 0.6611335072326298, 'invalid_json': 1, 'n': 452}


In [19]:
results = {"precision": absa_peft_test["p"], "recall": absa_peft_test["r"],
           "micro-f1":absa_peft_test["micro_f1"],
           "invalid_json": absa_peft_test["invalid_json"],
           "n": absa_peft_test["n"]}

In [20]:
results_dir = "/content/drive/MyDrive/FYP/absa/allam_lora_absa_peft/results"
os.makedirs(results_dir, exist_ok=True)

with open(os.path.join(results_dir, "result.json"), "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
